# Focused 3: Hide future answers, then look back at a separate source

**Optional returning-reader route.** The [canonical route](../README.md) remains the complete, authoritative route.

## Provenance

| Source notebook | Source headings copied | Revision |
|---|---|---|
| [04a-decoder-only-language-model-theory.ipynb](../04a-decoder-only-language-model-theory.ipynb) | Guiding question; 1. The dishonest shortcut; That fixed leakage, but three different things are called a mask; 2. One known sequence becomes many lessons; The cache saves work; it does not change the distribution; 3. One score vector, several decoding policies; Part 4A checkpoint | `e94a33f` |
| [05a-encoder-decoder-and-cross-attention-theory.ipynb](../05a-encoder-decoder-and-cross-attention-theory.ipynb) | Guiding question; 1. Attempt 1: squeeze the source into one mean; The mean knows what appeared, but not which address to retrieve; 2. Cross-attention keeps the roles separate; 4. Three reusable states, not one universal cache; Part 5A checkpoint | `e94a33f` |

Maintenance rule: **if canonical changes, update or mark stale**. This copy records the pushed baseline, not edits being made beside it.

Kernels do not share memory. Run this notebook from the top; it rebuilds all state locally.

## What you bring; what changes here

You know that a reader builds notes from a source and a writer predicts a separate answer. You also know practice can supply the correct previous answer token; that is teacher forcing.
For any missing piece, return to [Stage 03 model evolution](../../03-language-model-foundations/03-language-models/README.md) or [canonical seq2seq foundations](../../03-language-model-foundations/03-language-models/03A-training-skeleton-and-word-objectives.ipynb).

The new question is what each Transformer attention table may read. First keep future words off one growing tape. Then give the writer a second, fixed workspace to retrieve from. The setup creates both examples; no notebook imports another notebook.

In [ ]:
# Imports and deterministic teaching state
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

np.random.seed(404)
tokens = ["<bos>", "aria", "heard", "the", "signal", "<eos>"]
print("Running example:", " ".join(tokens))


source = np.array([3, 1, 4, 1])
target = source[::-1]
print("Separate source-to-target example:", source.tolist(), "->", target.tolist())

# Stage 04 Transformers, Part 4A: Why a Decoder-Only Model Works

## Guiding question

How can one stack learn every next-token lesson in parallel without peeking at the answers, then generate one token at a time without changing the objective?

Parts 1-3 gave Riverside's tokens identity, order, contextual routing, and a complete shape-preserving block. The unresolved complaint was architectural: a normal self-attention table can see both directions. A language model graded after `aria heard` must not inspect `the signal` first.

| Step | Visible failure | Smallest repair | Evidence |
|---|---|---|---|
| 1 | Full attention leaks future answers | Causal score mask | The forbidden triangle receives zero probability |
| 2 | One sentence looks like one lesson | Shift inputs and targets | Every valid position becomes one classification row |
| 3 | Batches need padding | Separate three mask jobs | Attention, padding, and loss exclusion remain distinct |
| 4 | Training is parallel; generation is serial | Feed choices back one at a time | Prefix work grows unless it is cached |
| 5 | Greedy output is brittle | Temperature and top-k | One logit vector yields controlled alternatives |

Keep [the handwritten theory companion](../04-decoder-only-language-model-theory.md) beside this notebook. Part 4B turns these contracts into a readable PyTorch implementation.

**Scope:** this notebook builds and measures the causal objective. It does not repeat the complete block from Part 3 or claim that a tiny corpus produces a useful assistant.


## 1. The dishonest shortcut

**Predict:** If the state after `heard` can read `signal`, will low next-token loss prove that the model learned to predict?

No. It may only prove that the answer was visible. The smallest repair is a lower-triangular visibility rule: row `i` may read columns `0..i`, never columns after `i`.


In [ ]:
# Static causal evidence: allowed positions survive, future positions disappear
length = len(tokens)
allowed = np.tril(np.ones((length, length), dtype=bool))
raw_scores = np.linspace(-0.8, 1.0, length * length).reshape(length, length)
masked_scores = np.where(allowed, raw_scores, -np.inf)
shifted = masked_scores - np.max(masked_scores, axis=-1, keepdims=True)
attention = np.exp(shifted)
attention = attention / attention.sum(axis=-1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, matrix, title in zip(
    axes,
    [allowed.astype(float), attention],
    ["Legal reads (1 = visible)", "Probability after causal masking"],
):
    image = ax.imshow(matrix, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(length), tokens, rotation=45, ha="right")
    ax.set_yticks(range(length), tokens)
    ax.set_xlabel("key position")
    ax.set_ylabel("query position")
    ax.set_title(title)
fig.colorbar(image, ax=axes, shrink=0.75)
plt.tight_layout()
plt.show()

future_mass = attention[np.triu_indices(length, k=1)].sum()
print(f"Total probability assigned to future positions: {future_mass:.1f}")
assert future_mass == 0.0


#### That fixed leakage, but three different things are called a mask

| Contract | Where it acts | What it blocks | Typical shape |
|---|---|---|---|
| Causal attention mask | attention score table | future key positions | `(time, time)` |
| Padding key mask | attention score table | placeholder keys in this batch | `(batch, time)` |
| Ignored loss target | cross-entropy rows | targets that are not real lessons | `(batch, time)` |

A causal mask cannot remove padded targets from the loss. `ignore_index` cannot stop attention from reading a padded key. Similar names do not make them interchangeable.


In [ ]:
# Shift once, then expose the three contracts on a padded two-example batch
examples = [
    ["<bos>", "aria", "heard", "the", "signal", "<eos>"],
    ["<bos>", "aria", "returned", "<eos>"],
]
width = max(len(row) - 1 for row in examples)
input_rows, target_rows, padding_rows = [], [], []
for row in examples:
    inputs, targets = row[:-1], row[1:]
    pad_count = width - len(inputs)
    input_rows.append(inputs + ["<pad>"] * pad_count)
    target_rows.append(targets + ["IGNORE"] * pad_count)
    padding_rows.append([False] * len(inputs) + [True] * pad_count)

print("inputs:")
for row in input_rows:
    print(" ", row)
print("targets:")
for row in target_rows:
    print(" ", row)
print("padding-key mask (True means block):", padding_rows[1])
print("causal mask is shared across the batch:", (width, width))


## 2. Same information boundary, different execution order

You already know shifted targets and teacher forcing; [canonical Part 4A](../04a-decoder-only-language-model-theory.ipynb) shows the full aligned-prefix explanation.
The Transformer-specific point is that a future-blind attention table can grade all known positions at once. Generation must still append its own last choice before computing the next one.
The next measured storyboard counts how much prefix work gets repeated. Keeping earlier keys and values, called a KV cache, saves work; it does not reveal future answers.

In [ ]:
# Measured recomputation complaint and a static storyboard for the cache
max_new_tokens = 6
prompt_length = 4
recomputed_positions = [
    prompt_length + step for step in range(max_new_tokens)
]
cached_positions = [prompt_length] + [1] * (max_new_tokens - 1)

print("positions processed without cache:", recomputed_positions)
print("positions processed with cache:   ", cached_positions)
print("total without cache:", sum(recomputed_positions))
print("total with cache:   ", sum(cached_positions))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
axes[0].bar(range(max_new_tokens), recomputed_positions, color="#c44e52")
axes[1].bar(range(max_new_tokens), cached_positions, color="#4c72b0")
axes[0].set_title("Recompute the whole prefix")
axes[1].set_title("Prime once, then append one state")
for ax in axes:
    ax.set_xlabel("generation step")
    ax.set_ylabel("positions processed")
plt.tight_layout()
plt.show()


In [ ]:
# Animation: the same measured work counts accumulate over generation
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.set_xlim(-0.5, max_new_tokens - 0.5)
ax.set_ylim(0, max(recomputed_positions) + 1)
ax.set_xlabel("generation step")
ax.set_ylabel("positions processed now")
bars_a = ax.bar(
    np.arange(max_new_tokens) - 0.18,
    np.zeros(max_new_tokens),
    width=0.36,
    label="recompute",
    color="#c44e52",
)
bars_b = ax.bar(
    np.arange(max_new_tokens) + 0.18,
    np.zeros(max_new_tokens),
    width=0.36,
    label="cache",
    color="#4c72b0",
)
ax.legend()

def update(frame):
    for index, bar in enumerate(bars_a):
        bar.set_height(recomputed_positions[index] if index <= frame else 0)
    for index, bar in enumerate(bars_b):
        bar.set_height(cached_positions[index] if index <= frame else 0)
    ax.set_title(f"Prefix grows through generation step {frame + 1}")
    return [*bars_a, *bars_b]

animation = FuncAnimation(fig, update, frames=max_new_tokens, interval=550, blit=False)
plt.close(fig)
display(HTML(animation.to_jshtml()))


#### The cache saves work; it does not change the distribution

Each layer stores earlier keys and values. The new token contributes one new query, key, and value. With the same weights, arithmetic, prefix, and selection rule, cached and full-prefix logits should agree within numerical tolerance. Part 4B tests that parity directly.

## 3. One score vector, several decoding policies

Greedy decoding always selects the largest logit. Sampling first reshapes the same scores:

- temperature below `1` sharpens preferences;
- temperature above `1` flattens them;
- top-k sets every candidate outside the best `k` to zero probability.

These controls change selection, not what the model learned.


In [ ]:
# One changed variable: temperature on the same four next-token logits
labels = ["signal", "map", "harbor", "lantern"]
logits = np.array([2.8, 1.6, 0.8, -0.2])

def probabilities(temperature):
    scaled = logits / temperature
    values = np.exp(scaled - scaled.max())
    return values / values.sum()

table = np.stack([probabilities(0.5), probabilities(1.0), probabilities(1.8)])
for temperature, row in zip([0.5, 1.0, 1.8], table):
    pairs = ", ".join(f"{label}={value:.3f}" for label, value in zip(labels, row))
    print(f"T={temperature:.1f}: {pairs}")

top_two = np.argsort(logits)[-2:]
print("top-k=2 keeps:", [labels[index] for index in top_two])


## Part 4A checkpoint

| Question | Evidence |
|---|---|
| Why causal masking? | Future-position probability measured exactly `0` |
| Why shift targets? | One sentence exposed five aligned next-token lessons |
| Why three mask contracts? | Score visibility, padded keys, and invalid targets act at different boundaries |
| Why serial generation? | Every new input depends on the prior selected token |
| Why cache? | The measured storyboard replaced repeated prefix work with one appended state |

**Built and measured:** causal visibility, aligned targets, mask distinctions, recomputation cost, temperature.

**Explained or illustrated:** top-k, stop tokens, per-layer KV cache.

**Named and deferred:** beam search and production cache layouts; they change search or systems engineering, not the causal objective.

**Unresolved question:** can a compact PyTorch model preserve every one of these contracts at once? Continue to [Part 4B: Decoder-Only PyTorch Lab](../04b-decoder-only-language-model-lab.ipynb).


## Change the microscope: one tape becomes two workspaces

The sentence showed why the writer must not peek ahead. Reversing `[3, 1, 4, 1]` now makes the source address visible without another tokenizer.
You already know the reader/writer roles. Here their internals become Transformer blocks: source self-attention lets source positions read one another; target self-attention hides future target tokens; cross-attention lets each target position read source notes.

```text
encoder:                self-attention -> per-token FFN
decoder-only:           future-blind self-attention -> per-token FFN
encoder-decoder writer: future-blind self-attention -> cross-attention -> per-token FFN
```

Each attention or FFN step also keeps its normalization and residual path. This sketch compares jobs, not a new block implementation.

The next mean-pooling attempt deliberately loses source addresses. It is not a learned recurrent summary: a recurrent reader can encode order before handing over its one note. For that controlled learned comparison, use [Stage 03 model evolution](../../03-language-model-foundations/03-language-models/README.md).
Keeping source positions fixes this particular access failure; it does not guarantee correct learned routing.

# Stage 04 Transformers, Part 5A: Why Cross-Attention Exists

## Guiding question

Why keep every source position if one pooled sentence vector is cheaper?

Decoder-only generation uses one growing tape. Translation, summarization, and structured transformation have two workspaces: read a complete source, then write a distinct target. The reversal diagnostic makes the required routing visible:

```text
source:  [3, 1, 4, 1]
target:  [1, 4, 1, 3]
```

| Complaint | Refinement | Evidence |
|---|---|---|
| One mean vector forgets which occurrence was where | Preserve one encoder vector per source position | Reordered sources collide under mean pooling |
| Decoder state and source memory have different roles | Q from decoder; K/V from encoder | Weight grid has shape `(target, source)` |
| Teacher-forced token scores look strong | Run free-running decoding | One early error changes every later prefix |
| Repeated source/prefix work is wasteful | Separate fixed and growing caches | Encoder and cross-KV stay fixed; self-KV grows |

Keep [the handwritten theory companion](../05-encoder-decoder-and-cross-attention-theory.md) beside this notebook. Part 5B implements the reversal microscope.


## Restore the source-routing state

The examples share a notebook, not a prediction task. Reset the source experiment's random seed so it matches its canonical notebook.

In [ ]:
# Restore the canonical source-routing seed
np.random.seed(505)
print("source:", source.tolist())
print("target:", target.tolist())

## 1. Attempt 1: squeeze the source into one mean

**Predict:** Can the mean distinguish `[3, 1, 4, 1]` from `[1, 4, 1, 3]`?

Mean pooling keeps content totals and discards arrangement. Adding one position vector to each token before taking the mean still sums the same set of positions, so the collision remains.


In [ ]:
# Measured bottleneck: two different orders collapse to the same summary
token_vectors = {
    1: np.array([1.0, 0.0, 0.2]),
    3: np.array([0.0, 1.0, 0.4]),
    4: np.array([0.2, 0.3, 1.0]),
}
position_vectors = np.eye(4, 3)

def pooled(sequence):
    states = np.stack(
        [token_vectors[token] + position_vectors[index] for index, token in enumerate(sequence)]
    )
    return states.mean(axis=0)

source_pool = pooled(source)
reordered_pool = pooled(target)
print("source mean:   ", np.round(source_pool, 4))
print("reordered mean:", np.round(reordered_pool, 4))
print("maximum difference:", float(np.abs(source_pool - reordered_pool).max()))
assert np.allclose(source_pool, reordered_pool)


#### The mean knows what appeared, but not which address to retrieve

Reversal needs a different source address at every target step. The smallest repair is not a larger pooled vector. It is an addressable memory with one row per source position.

## 2. Cross-attention keeps the roles separate

- decoder states form **queries**: what does this writing step need?
- encoder states form **keys**: what does each source position advertise?
- encoder states also form **values**: what information is delivered if selected?

The resulting table is `(target_length, source_length)`. Those lengths need not match. Cross-attention is not causally masked because the complete source is legal; it still needs a source padding mask.


In [ ]:
# Hand-built routing scores: each target step asks for the reversed source address
source_positions = np.arange(len(source))
target_steps = np.arange(len(target))
desired_positions = len(source) - 1 - target_steps
scores = -4.0 * (target_steps[:, None] * 0 + source_positions[None, :] - desired_positions[:, None]) ** 2
weights = np.exp(scores - scores.max(axis=-1, keepdims=True))
weights = weights / weights.sum(axis=-1, keepdims=True)

fig, ax = plt.subplots(figsize=(6, 4))
image = ax.imshow(weights, vmin=0, vmax=1, cmap="magma")
ax.set_xticks(source_positions, [f"src {i}: {token}" for i, token in enumerate(source)])
ax.set_yticks(target_steps, [f"step {i}: {token}" for i, token in enumerate(target)])
ax.set_xlabel("encoder key/value position")
ax.set_ylabel("decoder query position")
ax.set_title("Expected reversal routing")
fig.colorbar(image, ax=ax, label="attention weight")
plt.tight_layout()
plt.show()

routed = weights.argmax(axis=-1)
print("predicted source addresses:", routed.tolist())
assert np.array_equal(routed, desired_positions)


In [ ]:
# Animation: the anti-diagonal appears one decoder query at a time
fig, ax = plt.subplots(figsize=(6, 4))
shown = np.zeros_like(weights)
image = ax.imshow(shown, vmin=0, vmax=1, cmap="magma")
ax.set_xticks(source_positions, [str(value) for value in source])
ax.set_yticks(target_steps, [str(value) for value in target])
ax.set_xlabel("source token")
ax.set_ylabel("target token")

def update(frame):
    shown[frame] = weights[frame]
    image.set_data(shown)
    ax.set_title(f"decoder step {frame} retrieves source position {desired_positions[frame]}")
    return [image]

animation = FuncAnimation(fig, update, frames=len(target), interval=700, blit=False)
plt.close(fig)
display(HTML(animation.to_jshtml()))


## 3. Familiar training, a new source of evidence

Teacher forcing supplies the correct earlier target tokens; [canonical Part 5A](../05a-encoder-decoder-and-cross-attention-theory.ipynb) retains the full alignment recap.
The source notes stay available in both practice and generation. The target prefix does not: a wrong earlier choice changes later queries, which ask where to read next.
The next trace shows changed inputs, not proof that an error must cascade. Measure both token accuracy with correct history and whole-answer accuracy with the writer's own history; neither replaces the other.

In [ ]:
# Prefix lineage: one early mistake contaminates every later free-running input
gold = ["<bos>", "1", "4", "1", "3"]
free = ["<bos>", "1", "3", "1", "3"]
print("step | teacher-forced prefix | free-running prefix")
for step in range(1, len(gold)):
    teacher_prefix = " ".join(gold[:step])
    free_prefix = " ".join(free[:step])
    marker = "same" if teacher_prefix == free_prefix else "changed"
    print(f"{step:>4} | {teacher_prefix:<23} | {free_prefix:<23} ({marker})")


## 4. Three reusable states, not one universal cache

| Reusable state | Built from | Lifetime |
|---|---|---|
| Encoder memory | complete source | fixed for the request |
| Cross-attention K/V | layer-specific projections of encoder memory | fixed for the request |
| Decoder self-attention K/V | generated target prefix | grows one position per step |

Fixed encoder activations during inference do not mean frozen encoder parameters during training. Target loss normally backpropagates through cross-attention into the encoder.

## Part 5A checkpoint

**Built and measured:** mean-pooling collision, rectangular `(target, source)` routing, expected reversal anti-diagonal, prefix contamination.

**Explained or illustrated:** teacher forcing, exposure bias, fixed encoder and cross-KV caches, growing self-KV cache.

**Named and deferred:** beam search and production T5/BART demonstrations; they do not change the routing contract.

**Unresolved question:** will a learned `CrossAttention` module recover the predicted route and still generate without gold history? Continue to [Part 5B](../05b-encoder-decoder-and-cross-attention-lab.ipynb).


## Focused route checkpoint

| Question | Evidence to keep |
|---|---|
| May a next-token query read the next answer? | The forbidden triangle has zero probability |
| Does a padded rectangle contain real targets everywhere? | Visibility, empty-key exclusion, and target grading use different masks |
| Does caching change what may be read? | The storyboard changes repeated work, not the allowed history |
| Is one pooled source note the same as addressable memory? | Reordered sources collide under the mean; separate source positions retain addresses |
| Where do cross-attention queries and keys come from? | Writer queries meet source keys; the grid has target rows and source columns |

**Built and measured:** causal masking and leakage, shifted targets, three mask jobs, cache work counts and animation, temperature, pooling collision, cross-attention routing and animation, changed-prefix trace.

**Explained or illustrated:** reader/writer block differences, fixed source versus growing target states, top-k selection (keep only the highest-scoring candidates), practice versus generation evaluation.

**Named and deferred:** learned decoder and source-to-target training live in [Part 4B](../04b-decoder-only-language-model-lab.ipynb) and [Part 5B](../05b-encoder-decoder-and-cross-attention-lab.ipynb). Beam search compares several candidate continuations; production cache layouts organize saved states. They need their own search and systems experiments.

Hand-built scores reveal the routing contract, not a learned ability. Continue to those canonical labs for implementations, then [Part 6](../06-modern-decoder-only-llm.ipynb) for modern block refinements. The optional focused copy stops here; it never replaces the full route.